In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import torch
import pickle as pickle
import matplotlib.pyplot as plt
import pandas as pd
import time
import sys
sys.path.insert(0, '/data/env/VeloVAE')
import velovae as vv
import unitvelo as utv
import os.path
from os.path import exists
import time
import dynamo as dyn 
scv.settings.verbosity = 3
method = 'VeloVAE'

(Running UniTVelo 0.2.5.2)
2024-12-19 12:17:17


2024-12-19 07:17:17.360138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-19 07:17:17.384402: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-19 07:17:17.391726: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

In [ ]:

for dataset in datasets:
    print(dataset)
    model_path = f"/data_path/VeloVAE/checkpoints/{dataset}"
    data_path = f"/data_path/VeloVAE/data/{dataset}"
    figure_path = f'/data_path/VeloVAE/figures/{dataset}'
    adata = sc.read_h5ad(data_dir + dataset + f'/{dataset}.h5ad')
    start = time.time()
    scv.pp.filter_genes_dispersion(adata, n_top_genes=2000,log=False)
    scv.pp.moments(adata)
    dyn.tl.neighbors(adata,n_neighbors=30)
    torch.manual_seed(2022)
    np.random.seed(2022)
    vae = vv.VAE(adata, tmax=20, dim_z=5)
    vae.train(adata, plot=False, figure_path=figure_path, embed='umap')
    vae.train(adata,
            plot=False,
            figure_path=figure_path,
            embed='umap')
    end = time.time()
    vae.save_model(model_path, 'encoder_vae', 'decoder_vae')
    vae.save_anndata(adata, 'vae', data_path, file_name=f'{dataset}_out.h5ad')
    file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
    ground_truth = pickle.load(file)
    res, res_type = vv.post_analysis(adata,
                                    'continuous',
                                    methods= ['VeloVAE'],
                                    keys=['vae'],
                                    compute_metrics=False,
                                    raw_count=False,
                                    grid_size=(1,2),
                                    figure_path=figure_path,
                                    cluster_edges=ground_truth,
                                )
    # fix, ax = plt.subplots(1, 1, figsize = (8, 6))
    # dyn.pl.streamline_plot(adata, color=['clusters'], basis='umap', show_legend='on data', show_arrowed_spines=True,ax=ax)
    # plt.savefig(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.svg')
    # Calculate performance metrics:
    metrics = utv.evaluate(adata, ground_truth, 'clusters', 'vae_velocity')
    if exists(save_dir + method +'/'+ '_CBDir_scores.csv'):
        tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
    else:
        tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
        tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
    ##CBDC_scores
    cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
    tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
    df_CB = pd.concat([df_CB, pd.DataFrame([[np.mean(cb_score), end - start]], columns=df_CB.columns, index=[dataset])])
    tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
    ##ICCoh_scores
    IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                for x in metrics['In-cluster Coherence'].keys()]
    tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]
    df_IC = pd.concat([df_IC, pd.DataFrame([[np.mean(IC_score), end - start]], columns=df_IC.columns, index=[dataset])])
    tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
    
    adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [ ]:
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores.csv')